In [12]:
import os

os.environ["BRIGHT_API_KEY"] = "71da9406-409d-4066-8228-5b81aff2096c"
API_TOKEN = os.getenv("BRIGHT_API_KEY")

In [14]:
!pip install beautifulsoup4

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


In [ ]:
import os
import re
import json
import requests
import pandas as pd
from getpass import getpass
from html import unescape
from datetime import datetime


# ============================================================
# 1. 配置 Bright Data API Key
# ============================================================

# API_TOKEN = getpass("请输入 Bright Data API Key：").strip()

# # 防止复制时把 Bearer 一起复制进来
# if API_TOKEN.lower().startswith("bearer "):
#     API_TOKEN = API_TOKEN[7:].strip()

# # 去掉可能复制进去的引号、空格、换行
# API_TOKEN = API_TOKEN.strip('"').strip("'").strip()

# os.environ["BRIGHT_API_KEY"] = API_TOKEN

# print("API Key 已设置")
# print("Token 长度：", len(API_TOKEN))
# print("Token 后6位：", API_TOKEN[-6:])


# ============================================================
# 2. 商业化场景配置
# ============================================================
# 场景：AI 搜索品牌曝光监测
# 目标：模拟中小企业用户询问 ChatGPT 网络安全厂商推荐，
#      检测 ChatGPT 是否提到目标品牌和竞品。

DATASET_ID = "gd_m7aof0k82r803d5bjm"  # Bright Data ChatGPT Scraper

target_brand = "奇安信"

competitors = [
    "深信服",
    "启明星辰",
    "绿盟科技",
    "安恒信息",
    "天融信"
]

prompt = (
    "一家100人规模的电商公司想采购网络安全服务，"
    "请推荐3个中国网络安全厂商，不超过80字。"
)


# ============================================================
# 3. 调用 Bright Data ChatGPT Scraper
# ============================================================

api_url = (
    "https://api.brightdata.com/datasets/v3/scrape"
    f"?dataset_id={DATASET_ID}"
    "&format=json"
    "&notify=false"
    "&include_errors=true"
)

headers = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}

payload = [
    {
        "url": "https://chatgpt.com/",
        "prompt": prompt,
        "country": "",
        "web_search": False,
        "additional_prompt": ""
    }
]

response = requests.post(
    api_url,
    headers=headers,
    json=payload,
    timeout=120
)

print("状态码：", response.status_code)

if response.status_code != 200:
    print("请求失败，返回内容如下：")
    print(response.text[:1000])
    raise RuntimeError("请求失败，请检查 API Key、账户权限或 Scraper 是否开通")

print("请求成功")


# ============================================================
# 4. 解析 Bright Data 返回结果
# ============================================================

def html_to_text(html_content):
    """
    将 answer_html 转成纯文本。
    不依赖 BeautifulSoup，直接用正则和 html.unescape 处理。
    """
    if not html_content:
        return ""

    text = re.sub(r"<br\s*/?>", "\n", html_content, flags=re.IGNORECASE)
    text = re.sub(r"</p>|</li>|</div>|</h\d>", "\n", text, flags=re.IGNORECASE)
    text = re.sub(r"<[^>]+>", "", text)
    text = unescape(text)

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return "\n".join(lines)


def get_answer_text(item):
    """
    兼容不同返回字段：
    answer_text、answer_text_markdown、answer、answer_html
    """
    answer = (
        item.get("answer_text")
        or item.get("answer_text_markdown")
        or item.get("answer")
        or ""
    )

    if not answer and item.get("answer_html"):
        answer = html_to_text(item.get("answer_html"))

    return answer


data = response.json()

# 兼容 list / dict 两种返回
if isinstance(data, list):
    item = data[0]
elif isinstance(data, dict) and "data" in data and isinstance(data["data"], list):
    item = data["data"][0]
else:
    item = data

print("返回字段：", list(item.keys()))

answer = get_answer_text(item)

print("\n===== ChatGPT 回答 =====")
print(answer)


# ============================================================
# 5. 品牌曝光分析
# ============================================================

target_mentioned = target_brand in answer

mentioned_competitors = [
    brand for brand in competitors
    if brand in answer
]

# 判断目标品牌首次出现位置
target_position = answer.find(target_brand) if target_mentioned else -1

# 简单判断是否形成推荐
recommend_words = ["推荐", "适合", "选择", "可以考虑", "建议"]
is_recommended = target_mentioned and any(word in answer for word in recommend_words)

report = pd.DataFrame([
    {
        "商业场景": "中小企业网络安全采购",
        "使用的 LLM Scraper": "ChatGPT Scraper",
        "Bright Data Dataset ID": DATASET_ID,
        "用户问题": prompt,
        "ChatGPT 回答": answer,
        "目标品牌": target_brand,
        "是否提到目标品牌": "是" if target_mentioned else "否",
        "是否形成推荐": "是" if is_recommended else "否",
        "目标品牌首次出现位置": target_position if target_position != -1 else "未出现",
        "出现的竞品": "、".join(mentioned_competitors) if mentioned_competitors else "无",
        "竞品数量": len(mentioned_competitors),
        "是否触发联网搜索": item.get("web_search_triggered", "未知"),
        "使用模型": item.get("model", "未知")
    }
])

report

状态码： 202
请求失败，返回内容如下：
{"snapshot_id":"sd_mpk14axy1nhs5b20s3","message":"Your request is still in progress and cannot be retrieved in this call. Use the provided Snapshot ID to track progress via the Monitor Snapshot endpoint and download it once ready via the Download Snapshot endpoint. More information on https://docs.brightdata.com/api-reference/web-scraper-api/management-apis/monitor-progress"}


RuntimeError: 请求失败，请检查 API Key、账户权限或 Scraper 是否开通

In [17]:
import time
import re
import json
import requests
import pandas as pd
from html import unescape
from datetime import datetime


# ============================================================
# 1. 从刚刚 202 的 response 中取 snapshot_id
# ============================================================
# 你刚刚返回的是：
# {"snapshot_id":"sd_mpk14axy1nhs5b20s3", ...}

try:
    temp = response.json()
    snapshot_id = temp.get("snapshot_id")
except Exception:
    snapshot_id = None

# 如果上面的 response 变量因为报错中断不可用，就直接使用你刚刚返回的 snapshot_id
if not snapshot_id:
    snapshot_id = "sd_mpk14axy1nhs5b20s3"

print("当前 snapshot_id：", snapshot_id)


# ============================================================
# 2. 确保 headers 存在
# ============================================================
# 如果你前面的 headers 变量还在，这里会直接复用。
# 如果 headers 不存在，就用 API_TOKEN 重新构造。

try:
    headers
except NameError:
    headers = {
        "Authorization": f"Bearer {API_TOKEN}",
        "Content-Type": "application/json",
    }


# ============================================================
# 3. 轮询 snapshot 进度
# ============================================================

progress_url = f"https://api.brightdata.com/datasets/v3/progress/{snapshot_id}"

max_wait_seconds = 300
interval = 10
waited = 0

while True:
    progress_resp = requests.get(
        progress_url,
        headers=headers,
        timeout=60
    )

    print("进度查询状态码：", progress_resp.status_code)
    print("进度返回内容：", progress_resp.text[:500])

    if progress_resp.status_code != 200:
        raise RuntimeError("查询 snapshot 进度失败")

    progress_data = progress_resp.json()
    status = progress_data.get("status")

    print("当前任务状态：", status)

    if status == "ready":
        print("任务已完成，可以下载结果")
        break

    if status in ["failed", "error"]:
        raise RuntimeError(f"任务失败：{progress_data}")

    if waited >= max_wait_seconds:
        raise TimeoutError(
            f"等待超过 {max_wait_seconds} 秒，任务仍未完成。"
            f"你可以稍后继续用这个 snapshot_id 下载：{snapshot_id}"
        )

    time.sleep(interval)
    waited += interval


# ============================================================
# 4. 下载 snapshot 结果
# ============================================================

download_url = (
    f"https://api.brightdata.com/datasets/v3/snapshot/{snapshot_id}"
    "?format=json"
)

download_resp = requests.get(
    download_url,
    headers=headers,
    timeout=120
)

print("下载状态码：", download_resp.status_code)
print("下载返回预览：", download_resp.text[:500])

if download_resp.status_code != 200:
    raise RuntimeError("下载 snapshot 结果失败")

data = download_resp.json()


# ============================================================
# 5. 解析返回结果
# ============================================================

def html_to_text(html_content):
    """
    将 answer_html 转成纯文本。
    """
    if not html_content:
        return ""

    text = re.sub(r"<br\s*/?>", "\n", html_content, flags=re.IGNORECASE)
    text = re.sub(r"</p>|</li>|</div>|</h\d>", "\n", text, flags=re.IGNORECASE)
    text = re.sub(r"<[^>]+>", "", text)
    text = unescape(text)

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return "\n".join(lines)


def get_answer_text(item):
    """
    兼容 Bright Data 返回字段：
    answer_text、answer_text_markdown、answer、answer_html
    """
    answer = (
        item.get("answer_text")
        or item.get("answer_text_markdown")
        or item.get("answer")
        or ""
    )

    if not answer and item.get("answer_html"):
        answer = html_to_text(item.get("answer_html"))

    return answer


def normalize_result(data):
    """
    兼容 list / dict / data 字段几种结构。
    """
    if isinstance(data, list):
        return data

    if isinstance(data, dict):
        if "data" in data and isinstance(data["data"], list):
            return data["data"]
        return [data]

    return []


records = normalize_result(data)

if not records:
    raise RuntimeError("没有解析到任何返回记录")

item = records[0]

print("\n返回字段：")
print(list(item.keys()))

answer = get_answer_text(item)

print("\n===== ChatGPT 回答 =====")
print(answer)


# ============================================================
# 6. 品牌曝光分析
# ============================================================

target_brand = "奇安信"

competitors = [
    "深信服",
    "启明星辰",
    "绿盟科技",
    "安恒信息",
    "天融信"
]

try:
    prompt
except NameError:
    prompt = (
        "一家100人规模的电商公司想采购网络安全服务，"
        "请推荐3个中国网络安全厂商，不超过80字。"
    )

target_mentioned = target_brand in answer

mentioned_competitors = [
    brand for brand in competitors
    if brand in answer
]

target_position = answer.find(target_brand) if target_mentioned else -1

recommend_words = ["推荐", "适合", "选择", "可以考虑", "建议"]
is_recommended = target_mentioned and any(word in answer for word in recommend_words)

report = pd.DataFrame([
    {
        "商业场景": "中小企业网络安全采购",
        "使用的 LLM Scraper": "ChatGPT Scraper",
        "Bright Data Snapshot ID": snapshot_id,
        "用户问题": prompt,
        "ChatGPT 回答": answer,
        "目标品牌": target_brand,
        "是否提到目标品牌": "是" if target_mentioned else "否",
        "是否形成推荐": "是" if is_recommended else "否",
        "目标品牌首次出现位置": target_position if target_position != -1 else "未出现",
        "出现的竞品": "、".join(mentioned_competitors) if mentioned_competitors else "无",
        "竞品数量": len(mentioned_competitors),
        "是否触发联网搜索": item.get("web_search_triggered", "未知"),
        "使用模型": item.get("model", "未知")
    }
])

report

当前 snapshot_id： sd_mpk14axy1nhs5b20s3
进度查询状态码： 200
进度返回内容： {"status":"ready","snapshot_id":"sd_mpk14axy1nhs5b20s3","dataset_id":"gd_m7aof0k82r803d5bjm","records":1,"errors":0,"collection_duration":135438,"avg_duration_per_input":135438}
当前任务状态： ready
任务已完成，可以下载结果
下载状态码： 200
下载返回预览： [
  {
    "url": "https://chatgpt.com/?q=%E4%B8%80%E5%AE%B6100%E4%BA%BA%E8%A7%84%E6%A8%A1%E7%9A%84%E7%94%B5%E5%95%86%E5%85%AC%E5%8F%B8%E6%83%B3%E9%87%87%E8%B4%AD%E7%BD%91%E7%BB%9C%E5%AE%89%E5%85%A8%E6%9C%8D%E5%8A%A1%EF%BC%8C%E8%AF%B7%E6%8E%A8%E8%8D%903%E4%B8%AA%E4%B8%AD%E5%9B%BD%E7%BD%91%E7%BB%9C%E5%AE%89%E5%85%A8%E5%8E%82%E5%95%86%EF%BC%8C%E4%B8%8D%E8%B6%85%E8%BF%8780%E5%AD%97%E3%80%82",
    "prompt": "一家100人规模的电商公司想采购网络安全服务，请推荐3个中国网络安全厂商，不超过80字。",
    "answer_html": "<html lang=\"en-US\" data-

返回字段：
['url', 'prompt', 'answer_html', 'answer_text', 'links_attached', 'citations', 'recommendations', 'country', 'is_map', 'references', 'shopping', 'shopping_visible', 'index', 'answer_text_markdown', 'web_search

,商业场景,使用的 LLM Scraper,Bright Data Snapshot ID,用户问题,ChatGPT 回答,目标品牌,是否提到目标品牌,是否形成推荐,目标品牌首次出现位置,出现的竞品,竞品数量,是否触发联网搜索,使用模型
0,中小企业网络安全采购,ChatGPT Scraper,sd_mpk14axy1nhs5b20s3,一家100人规模的电商公司想采购网络安全服务，请推荐3个中国网络安全厂商，不超过80字。,推荐： 奇安信 、 深信服 、 安恒信息 。适合100人电商企业，覆盖终端、防火墙与数据安全。,奇安信,是,是,4,深信服、安恒信息,2,False,gpt-5-5


In [11]:
DATASET_ID = "gd_m7aof0k82r803d5bjm"

api_url = (
    "https://api.brightdata.com/datasets/v3/scrape"
    f"?dataset_id={DATASET_ID}"
    "&format=json"
    "&notify=false"
    "&include_errors=true"
)

headers = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}

prompt = (
    "一家100人规模的电商公司想采购网络安全服务，"
    "请推荐3个中国网络安全厂商，不超过80字。"
)

payload = [
    {
        "url": "https://chatgpt.com/",
        "prompt": prompt,
        "country": "",
        "web_search": False,
        "additional_prompt": ""
    }
]

response = requests.post(
    api_url,
    headers=headers,
    json=payload,
    timeout=90
)

print("状态码：", response.status_code)
print(response.text[:800])

状态码： 401
Invalid credentials


In [ ]:
if response.status_code != 200:
    raise RuntimeError("请求失败，请检查 API Key、账户权限或 Scraper 是否已开通")

data = response.json()

item = data[0] if isinstance(data, list) else data

answer = (
    item.get("answer_text")
    or item.get("answer_text_markdown")
    or item.get("answer")
    or ""
)

target_brand = "奇安信"

competitors = [
    "深信服",
    "启明星辰",
    "绿盟科技",
    "安恒信息",
    "天融信"
]

target_mentioned = target_brand in answer

mentioned_competitors = [
    brand for brand in competitors
    if brand in answer
]

report = pd.DataFrame([
    {
        "商业场景": "中小企业网络安全采购",
        "使用的LLM Scraper": "ChatGPT Scraper",
        "用户问题": prompt,
        "ChatGPT回答": answer,
        "目标品牌": target_brand,
        "是否提到目标品牌": "是" if target_mentioned else "否",
        "出现的竞品": "、".join(mentioned_competitors) if mentioned_competitors else "无",
        "竞品数量": len(mentioned_competitors),
        "是否触发联网搜索": item.get("web_search_triggered", "未知"),
        "使用模型": item.get("model", "未知")
    }
])

report